# Multimodal Large Language Models

In [1]:
import sys
import os

# Adds the project root (one level up if running from notebooks/) to sys.path
sys.path.append(os.path.abspath(".."))  

from torch_geometric.loader import DataLoader
from src.dataloader.load_real_data import prepare_metr_la_datasets

datasets = prepare_metr_la_datasets("../data/raw/METR-LA/metr_la.h5", "../data/raw/METR-LA/adj_mx.pkl")
train_loader = DataLoader(datasets["train"], batch_size=32, shuffle=True)

batch = next(iter(train_loader))
print(f"Batch x shape: {batch['x'].shape}")  # Shape: [32, 12, 207, 1]
print(f"Batch y shape: {batch['y'].shape}")  # Shape: [32, 12, 207, 1]

Batch x shape: torch.Size([32, 12, 207, 1])
Batch y shape: torch.Size([32, 12, 207, 1])


/home/ubuntu/projects/MLLMs/src/dataloader/load_real_data.py:44: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  pickle_data = pickle.load(f, encoding='latin1')


# An example

In [3]:
import io
import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# 1. Generate sample time-series data (e.g., IoT Sensor Stream)
data = {
    'time': range(24),
    'sensor_value': [12, 14, 13, 15, 28, 45, 52, 48, 30, 22, 19, 18, 
                     20, 22, 25, 24, 29, 46, 60, 55, 40, 28, 18, 14]
}
df = pd.DataFrame(data)

In [4]:
# 2. Render time-series into a chart image (Modality Bridge)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df['time'], df['sensor_value'], marker='o', color='teal', label='Sensor Load')
ax.set_title("24-Hour Industrial Sensor Stream")
ax.set_xlabel("Time (Hours)")
ax.set_ylabel("Reading Magnitude")
ax.grid(True)

image_path = "timeseries_chart.png"
plt.savefig(image_path, bbox_inches='tight')
plt.close()

In [5]:
# 3. Load local Qwen2-VL-7B-Instruct
model_id = "Qwen/Qwen2-VL-7B-Instruct"
print(f"Loading local model: {model_id}...")

# Load processor and model ( utilizing half-precision float16 for memory efficiency )
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

Loading local model: Qwen/Qwen2-VL-7B-Instruct...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [6]:
from huggingface_hub import scan_cache_dir

# Scans your default Hugging Face cache
hf_cache = scan_cache_dir()

for repo in hf_cache.repos:
    if "Qwen2-VL" in repo.repo_id:
        print(f"Found Model: {repo.repo_id}")
        print(f"Storage Size: {repo.size_on_disk_str}")
        print(f"Path: {repo.repo_path}")

Found Model: Qwen/Qwen2-VL-7B-Instruct
Storage Size: 16.6G
Path: /home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2-VL-7B-Instruct


In [7]:
# 4. Construct messages using Qwen's standard chat template format
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {
                "type": "text", 
                "text": "Analyze this time series chart. Identify the specific hours where major spikes occur, and explain the overall trend sequence."
            },
        ],
    }
]

In [8]:
# 5. Prepare inputs using Qwen processor utils
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
# 6. Generate inference response
print("Running inference with Qwen2-VL...")
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=200)

# Trim input prompt tokens from output sequence
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print("\n--- Qwen2-VL Time-Series Analysis Result ---")
print(output_text)

Running inference with Qwen2-VL...

--- Qwen2-VL Time-Series Analysis Result ---
This is a line graph titled 24-Hour Industrial Sensor Stream. The y-axis measures Reading Magnitude while the x-axis shows Time (Hours). The graph shows a spike in reading magnitude at around 5 hours, followed by a sharp decline. There is another spike at around 10 hours, followed by a gradual decline. The reading magnitude then increases sharply again at around 15 hours, followed by a gradual decline. Overall, the trend shows a general increase in reading magnitude over the 24-hour period, with two major spikes and a few smaller fluctuations.


# Save Time with Quantization (Loss Accuracy!)

In [13]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# ==========================================
# STEP 1: Generate Mock Time-Series Chart
# ==========================================
print("Generating time-series chart...")
data = {
    'time': range(24),
    'sensor_value': [12, 14, 13, 15, 28, 45, 52, 48, 30, 22, 19, 18, 
                     20, 22, 25, 24, 29, 46, 60, 55, 40, 28, 18, 14]
}
df = pd.DataFrame(data)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df['time'], df['sensor_value'], marker='o', color='teal', label='Sensor Load')
ax.set_title("24-Hour Industrial Sensor Stream")
ax.set_xlabel("Time (Hours)")
ax.set_ylabel("Reading Magnitude")
ax.grid(True)

image_path = "timeseries_chart.png"
plt.savefig(image_path, bbox_inches='tight')
plt.close()

# ==========================================
# STEP 2: Corrected 4-Bit Quantization Setup
# ==========================================
# ==========================================
# STEP 2: Fixed 4-Bit Quantization (Fits inside 8GB VRAM)
# ==========================================
from transformers import BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-7B-Instruct"
print(f"Loading local model {model_id} with strict 4-bit GPU configuration...")

# Configure 4-bit quantization explicitly bound to cuda:0
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(model_id)

# Force the entire model to map strictly onto cuda to avoid CPU offloading crashes
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0}  # Forces all layers into GPU 0 cleanly
)

# ==========================================
# STEP 3: Construct Multimodal Prompt
# ==========================================
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {
                "type": "text", 
                "text": "Analyze this time series chart. Identify the specific hours where major spikes occur, and explain the overall trend sequence."
            },
        ],
    }
]

# Format text via chat template
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# Process images and text into tensors
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# STEP 4: Run Inference
# ==========================================
print("Running inference with Qwen2-VL...")
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=200)

# Trim prompt tokens and decode response
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print("\n" + "="*40)
print("QWEN2-VL TIME-SERIES ANALYSIS RESULT:")
print("="*40)
print(output_text)

Generating time-series chart...
Loading local model Qwen/Qwen2-VL-7B-Instruct with strict 4-bit GPU configuration...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Running inference with Qwen2-VL...

QWEN2-VL TIME-SERIES ANALYSIS RESULT:
This is a line graph titled 24-Hour Industrial Sensor Stream. The x-axis plots Time (Hours) while the y-axis plots Reading Magnitude. The graph shows a spike in reading magnitude at around 10 hours, followed by a sharp decline. There is another spike around 15 hours, followed by a gradual decline. The overall trend is a peak around 20 hours, followed by a decline.


# Using Quantization 

(has less information!)

Running inference with Qwen2-VL...
```text
========================================
QWEN2-VL TIME-SERIES ANALYSIS RESULT:
========================================
```

This is a line graph titled 24-Hour Industrial Sensor Stream. The x-axis plots Time (Hours) while the y-axis plots Reading Magnitude. The graph shows a spike in reading magnitude at around 10 hours, followed by a sharp decline. There is another spike around 15 hours, followed by a gradual decline. The overall trend is a peak around 20 hours, followed by a decline.

# Normal Execution 

(has more information!)

Running inference with Qwen2-VL...

--- Qwen2-VL Time-Series Analysis Result ---
This is a line graph titled 24-Hour Industrial Sensor Stream. The y-axis measures Reading Magnitude while the x-axis shows Time (Hours). The graph shows a spike in reading magnitude at around 5 hours, followed by a sharp decline. There is another spike at around 10 hours, followed by a gradual decline. The reading magnitude then increases sharply again at around 15 hours, followed by a gradual decline. Overall, the trend shows a general increase in reading magnitude over the 24-hour period, with two major spikes and a few smaller fluctuations.